# Pairwise Interaction Tracking Experiment


## Notes
**Overview:**   
**Rationale:**     
**Date of Experiment:**      
**Genotypes:**    
**Sex:**    
**Light Paradigm:**    
**Design:**    
**Misc:** 

## Configuration
Configuration was standard. No new or untested code.

In [ ]:
#%autoreload 1
import warnings
from Arena import *
import Tracker
import PairwiseInteractionTracker
from ExperimentalDesign import *
from Parameters import *

# Suppress all warnings because Pandas gives alot of them.
warnings.filterwarnings('ignore')

## Parameters
Use a member function of the parameters class to setup up a standard arena max experiment, accepting the default values and specifying a PAIRWISEINTERACTIONTRACKER by default. Interaction distances must be a list, although if you just want to consider one distance (e.g., 8mm) as interacting, you can specify that as    

`distances = [8]`.

In [ ]:
p=Parameters()
## Set whatever interaction distances (in mm) that you want to examine here:
distances=[4,8,10]
#p.set_pairwise_interaction_values_small_arena(interaction_distances=distances)
p.set_pairwise_interaction_values_arena_max(interaction_distances=distances)
p.print()

## Read in the data

Next we will read in the data to an Arena object. The format is    

`Arena(ExperimentName, parameters, data_path="./")`.   

This will automatically calculate all relevant metrics for the tracking type in question.  

The name of the experiment and predefined parameters object are required, but _data_path_ is optional (if no data directory is specified, it defaults to the current working directory).

The following files must be present in the defined data directory :
* The DTrack experiment file, which is an excel sheet with the name '_ExperimentName.xlsx_'.
* The DTrack data files themselves, which are named '_ExperimentName_Data_1.csv_', '_ExperimentName_Data_2.csv_', etc.
* An experiment file, which is named '_ExperimentName_Design.txt_'. Note that you can incorporate flipped axes to orient lights for data visualization.

In [ ]:
arena = Arena.Arena('MaxxxPWI_FLIR',p,data_path='./Data/')

## Data Summary
Get the summary results for all trackers, including treatments, etc.   

This will copy the results to the clipboard and save the data in a file called '_ExperimentName_Summary.csv_'. It will be saved in the specified data directory.      

The semicolon at the end of the command will suppress the output to this notebook.  Remove it if you want to see the resulting dataframe.

In [ ]:
arena.summarize(copy_to_clipboard=True,write_to_csvfile=True);

Or you can also create a faceted summary if you want to separate the different phases of the experiment.   

The file name will be '_ExperimentName_Summary_Facet.csv_'.

In [ ]:
## In this case the experiment started with 10min of lights off, then 60min of light treatment, then 10min of lights off again.
arena.summarize_facet(cutoffs=(10,70),copy_to_clipboard=True,write_to_csvfile=True);

Note that you can ask for the summary including only one of the two partner flies in each tracking region.  This is helpful for plotting, mostly to avoid double points.  In general you should get summary data for all flies to examine movement, etc.   Parenters will be automatically removed for treatment plotting and for statistical comparison functions.     
   
`arena.summarize(remove_partners=True)`

**Note**: For counting experiments, there are no individualized tracking data, so regions are summarized by the average values obtained from pseudo trackers in each region. It therefore does not make sense to plot data by tracker. However, one can plot the interactions as a function of time for each tracking region and can examine all of the positions for all of the flies pooled in each frame.  So plots are provided for each, although their interpretation is somewhat dicey.

## Plotting
There are many useful plotting funtions.  Some are shown below. If you have an idea for one, let me know.

**Facet plots:** One for each defined value of interaction distance and then one for Total Distance (in mm/min).   

In this example, the cutoffs parameter separates the acclimaton phase from the experiment phase from the cool down phase.

In [ ]:
arena.plot_interactions_facet(cutoffs=(10,70))

The pairwise comparisons function will automatically remove partners when the metric is PercentInteracting or FramesInteracting. This will avoid double counting individual chambers, which is the experimental unit in these experiments.  Partners may not be removed if you analyze a column that is individual, such as totaldistance.

In [ ]:

## This should iterate through all defined interaction distances.
for dist in arena.parameters.interaction_distance_mm:
    arena.run_pairwise_comparisons_facet(metric="PercentInteracting_"+str(dist),cutoffs=(10,70))

In [ ]:
## Here is the facet of TotalDistance as promised.  Most often this declines with time in the arena.
arena.plot_totaldistance_facet(cutoffs=(10,70))

**Statistical analyses**: This will be done with respect to treatments. T-tests will be used for designs with two treatments and Tukey multiple comparisons will be done if there are more than two treatments.

In [ ]:
arena.run_pairwise_comparisons_facet(metric="TotalDistance",cutoffs=(10,70))

**Tracker plots:** These show, for each tracker individually, the average interaction over windows of time and each position of the tracker over the experimenyt time.  As expected, for the time dependent interaction plots, partners are removed so that there is only one plot per chamber.  For thecoordinate plots, each separate tracker/fly gets a plot.

In [ ]:
arena.plot_trackers_time_dependent_interactions()

**Positional Plots:** Show the X positions of each tracker as a function of time.

In [ ]:
## These plots will present histograms of all the x (or y) positions of all the flies over the perscribed range of time (if provided).
arena.plot_trackers_x()
arena.plot_trackers_y(range_minutes=(10,70))    

In [ ]:
## This will just plop all frame positions for all flies over the range of time requested.
arena.plot_trackers_xy()